# 17. k-Nearest Neighbors (k-NN)
**Use case:** Supply Chain — Delivery transit-day estimation from similar past shipments

Customers expect an accurate delivery date at checkout, but transit time depends on distance, weight, carrier, season and service level. Planners today use a static miles-per-day table, so promises are missed in peak season and over-padded the rest of the year. The transportation management system holds years of shipment history with actual transit days. Success means estimates within one day for most shipments, a promise date that hits about 90% on-time, and an explanation customer service can show ('similar shipments took 8 days').

> 💡 Pricing a house by asking: what did the three most similar houses on this street sell for? You do not build a formula - you just find the closest comparables and average them. Estate agents have done k-NN for decades.

Legend in code: `# 🐍 PY:` Python lesson · `# 🧠 ML:` ML lesson

In [ ]:
# Run once (Colab already has most of these)
!pip -q install numpy pandas scikit-learn scipy xgboost torch

## EASY: Find the 3 most similar past shipments by hand
Compute distances from a new shipment to 8 past shipments in plain Python and average the transit days of the nearest ones.

| 5W + H | |
|---|---|
| **Who** | A logistics planner who quotes delivery dates to customers. |
| **What** | Estimate transit days for a new 1,000-mile, 1,000-lb shipment from 8 past shipments. |
| **When** | At order time, when the customer asks 'when will it arrive?'. |
| **Where** | A small Python script - no ML library, just the math module. |
| **Why** | Shipments that look alike (similar distance and weight) usually take a similar number of days - that is the whole idea of k-NN. |
| **How** | Scale distance and weight by their max, compute Euclidean distance to each past shipment, sort, and average the k closest (plain and distance-weighted). |

**🧠 ML concepts:** Instance-based (lazy) learning, Euclidean distance, Why features must be on similar scales, Choosing k, Distance weighting  
**🐍 Python concepts:** List of dicts, Generator expression inside max() and sum(), Defining a function with a docstring, sorted with key=lambda, Slicing [:k], enumerate

In [ ]:
# Easy: estimate transit days for a new shipment by finding the most similar past shipments
import math

# 🐍 PY: a list of dicts - each dict is one past shipment with named fields
past = [
    {"id": "S1", "distance_mi": 300,  "weight_lb": 500,  "transit_days": 2},
    {"id": "S2", "distance_mi": 450,  "weight_lb": 1200, "transit_days": 3},
    {"id": "S3", "distance_mi": 900,  "weight_lb": 800,  "transit_days": 4},
    {"id": "S4", "distance_mi": 1200, "weight_lb": 2000, "transit_days": 5},
    {"id": "S5", "distance_mi": 1500, "weight_lb": 600,  "transit_days": 5},
    {"id": "S6", "distance_mi": 2000, "weight_lb": 1500, "transit_days": 7},
    {"id": "S7", "distance_mi": 2400, "weight_lb": 3000, "transit_days": 8},
    {"id": "S8", "distance_mi": 600,  "weight_lb": 2500, "transit_days": 3},
]
new = {"distance_mi": 1000, "weight_lb": 1000}

# 🧠 ML: miles and pounds have different ranges, so divide each by its max to put both on 0-1
# 🐍 PY: max() with a generator expression finds the biggest value without building a list
max_d = max(s["distance_mi"] for s in past)
max_w = max(s["weight_lb"] for s in past)

def distance(a, b):
    """Euclidean distance between two shipments on the scaled features."""
    dd = (a["distance_mi"] - b["distance_mi"]) / max_d
    dw = (a["weight_lb"] - b["weight_lb"]) / max_w
    # 🧠 ML: straight-line (Euclidean) distance = sqrt of the summed squared differences
    return math.sqrt(dd ** 2 + dw ** 2)

# 🐍 PY: add a new key to every dict in place
for s in past:
    s["dist"] = distance(new, s)

# 🐍 PY: sorted() with key=lambda sorts dicts by one field
ranked = sorted(past, key=lambda s: s["dist"])
print("Past shipments ranked by similarity to the new one:")
for s in ranked:
    print(f"  {s['id']}  {s['distance_mi']:>5} mi  {s['weight_lb']:>5} lb  "
          f"days={s['transit_days']}  distance={s['dist']:.3f}")

# 🧠 ML: k-NN prediction = average the target of the k closest neighbours
for k in (1, 3, 5):
    # 🐍 PY: slicing [:k] takes the first k items of the sorted list
    nearest = ranked[:k]
    avg = sum(s["transit_days"] for s in nearest) / k
    ids = ", ".join(s["id"] for s in nearest)
    print(f"k={k}: neighbours [{ids}] -> predicted transit {avg:.2f} days")

# 🧠 ML: distance-weighted version - closer neighbours get a bigger vote (weight = 1 / distance)
k = 3
weights = [1 / ranked[i]["dist"] for i in range(k)]
weighted = sum(w * ranked[i]["transit_days"] for i, w in enumerate(weights)) / sum(weights)
print(f"k=3 distance-weighted -> {weighted:.2f} days")

**Try this next:**
- Change the new shipment to 2,200 miles and 2,800 lb - which neighbours are picked now?
- Remove the scaling (divide by 1 instead of max_d/max_w) and see how weight starts to dominate the distance.
- Add a 'carrier_delay' field and include it in the distance function.

## MEDIUM: KNeighborsRegressor on 3,000 shipments
Train scikit-learn's KNeighborsRegressor on scaled shipment features, compare k values and scaling, and explain a prediction with its neighbours.

| 5W + H | |
|---|---|
| **Who** | A transportation analytics team at a shipper or 3PL. |
| **What** | Predict transit days from distance, weight and season flags (peak, winter). |
| **When** | When a shipment is booked, to set a customer-facing ETA. |
| **Where** | A batch model refreshed weekly from the transportation management system history. |
| **Why** | Rule tables (miles / 500 = days) ignore weight and seasonal congestion; a neighbour-based estimate reflects what really happened to similar loads. |
| **How** | Generate 3,000 synthetic shipments, split, scale with StandardScaler, compare k and scaled vs unscaled MAE against a mean baseline, then use kneighbors to show the 5 most similar past shipments. |

**🧠 ML concepts:** k-NN regression, Bias-variance trade-off through k, Feature scaling for distance models, Baseline comparison, MAE and R-squared, Neighbour-based explanation  
**🐍 Python concepts:** numpy vectorised column creation, np.isin and astype, pandas describe and loc, Looping over tuples of arrays, DataFrame.assign, iloc indexing

In [ ]:
# Medium: KNeighborsRegressor predicting transit days from distance, weight and season
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_absolute_error, r2_score

np.random.seed(42)
N = 3000

# 🐍 PY: build columns as numpy arrays, then wrap them in a DataFrame
distance_mi = np.random.uniform(50, 2800, N)
weight_lb = np.random.lognormal(mean=7, sigma=0.6, size=N)
# 🐍 PY: np.random.choice samples month numbers 1-12
ship_month = np.random.choice(np.arange(1, 13), N)
is_peak = np.isin(ship_month, [11, 12]).astype(int)
is_winter = np.isin(ship_month, [1, 2, 12]).astype(int)

# True process: ~1 day per 500 miles, heavy freight slower, peak season and winter add delays
transit_days = (1 + distance_mi / 500 + 0.0002 * weight_lb
                + 1.2 * is_peak + 0.6 * is_winter + np.random.normal(0, 0.6, N))
df = pd.DataFrame({
    "distance_mi": distance_mi.round(0), "weight_lb": weight_lb.round(0),
    "is_peak": is_peak, "is_winter": is_winter,
    # 🐍 PY: np.clip keeps values within [1, inf) and .round(1) tidies decimals
    "transit_days": np.clip(transit_days, 1, None).round(1),
})
print(df.head().to_string(), "\n")
print(df.describe().loc[["mean", "min", "max"]].round(1).to_string(), "\n")

# 🐍 PY: a list of column names selects several DataFrame columns at once
features = ["distance_mi", "weight_lb", "is_peak", "is_winter"]
X_train, X_test, y_train, y_test = train_test_split(
    df[features], df["transit_days"], test_size=0.2, random_state=42)

# 🧠 ML: k-NN is pure distance - without scaling, miles (thousands) would drown out the 0/1 season flags
scaler = StandardScaler().fit(X_train)
X_train_s, X_test_s = scaler.transform(X_train), scaler.transform(X_test)

# 🧠 ML: baseline = always predict the training average; a model must beat this to be useful
base_mae = mean_absolute_error(y_test, np.full(len(y_test), y_train.mean()))
print(f"Baseline (predict mean) MAE: {base_mae:.2f} days")

print("\n  k | MAE unscaled | MAE scaled")
for k in [1, 5, 15, 40]:
    # 🐍 PY: a tuple of (train, test) pairs lets one loop body handle both versions
    maes = []
    for Xtr, Xte in ((X_train, X_test), (X_train_s, X_test_s)):
        knn = KNeighborsRegressor(n_neighbors=k).fit(Xtr, y_train)
        maes.append(mean_absolute_error(y_test, knn.predict(Xte)))
    print(f"{k:>3} |    {maes[0]:.2f}      |   {maes[1]:.2f}")

# 🧠 ML: small k = noisy (high variance); large k = over-smoothed (high bias); pick a middle value
model = KNeighborsRegressor(n_neighbors=15).fit(X_train_s, y_train)
pred = model.predict(X_test_s)
print(f"\nChosen k=15  MAE {mean_absolute_error(y_test, pred):.2f} days   R2 {r2_score(y_test, pred):.3f}")

# Score a brand-new shipment
new = pd.DataFrame([{"distance_mi": 1400, "weight_lb": 2200, "is_peak": 1, "is_winter": 1}])
est = model.predict(scaler.transform(new))[0]
# 🧠 ML: kneighbors returns the actual past shipments used - built-in explainability
dist, idx = model.kneighbors(scaler.transform(new), n_neighbors=5)
print(f"\nNew shipment 1400 mi, 2200 lb, December -> estimate {est:.1f} days")
print("Five closest past shipments:")
print(X_train.iloc[idx[0]].assign(transit_days=y_train.iloc[idx[0]].values).to_string())

# Business interpretation
within_1 = np.mean(np.abs(pred - y_test) <= 1.0)
print(f"\n{within_1:.0%} of test shipments predicted within +/- 1 day -> good enough to quote customer ETAs.")

**Try this next:**
- Set weights='distance' in KNeighborsRegressor and compare MAE.
- Remove is_peak from features and see what happens to the December prediction.
- Change N to 300 and observe how k-NN suffers with less history.

## HARD: Tuned k-NN pipeline with carriers, seasons and promise dates
Build a ColumnTransformer + KNeighborsRegressor pipeline, tune k, weights and metric, compare against baselines on a time-ordered test set, and set a buffered promise date.

| 5W + H | |
|---|---|
| **Who** | The supply chain data science team owning delivery-date promises for an e-commerce or B2B shipper. |
| **What** | A production ETA model using distance, weight, carrier, season and service level, with neighbour explanations for customer service. |
| **When** | At checkout or booking for every shipment, refit monthly as carrier performance drifts. |
| **Where** | An ETA microservice called by the order management system; model artifact saved with joblib. |
| **Why** | Promise dates that are too optimistic cause complaints and chargebacks; too pessimistic loses sales. A tuned model plus an error buffer hits a target on-time rate. |
| **How** | One-hot encode categories, scale numbers, GridSearchCV over k, weights and p, train on older shipments and test on the newest, compare with mean and linear baselines, use permutation importance, explain neighbours, and pick a 90th-percentile buffer. |

**🧠 ML concepts:** ColumnTransformer for mixed data types, One-hot encoding inside a distance model, Hyperparameter tuning of k, weights and Minkowski p, Time-ordered validation, Permutation importance, Quantile-based safety buffer  
**🐍 Python concepts:** Module-level constants and dicts, np.select as vectorised if/elif, Series.map with a dict, Functions returning DataFrames, named_steps on a Pipeline, Dict of models looped with items()

In [ ]:
# Hard: production transit-day estimator - one-hot + scaling pipeline, tuned k-NN, explainable neighbours
import os
import tempfile
import warnings
import joblib
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.neighbors import KNeighborsRegressor
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import GridSearchCV, KFold
from sklearn.inspection import permutation_importance
from sklearn.metrics import mean_absolute_error

warnings.filterwarnings("ignore")
rng = np.random.default_rng(42)
NUM = ["distance_mi", "weight_lb"]
CAT = ["carrier", "season", "service"]
CARRIER_DELAY = {"CarrierA": 0.0, "CarrierB": 0.5, "CarrierC": 1.2, "CarrierD": -0.3}
SEASON_DELAY = {"Winter": 0.7, "Spring": 0.0, "Summer": 0.2, "Peak": 1.4}


def make_shipments(n=4000):
    """Synthetic shipment history with carrier, season and service level effects."""
    df = pd.DataFrame({
        "ship_date": pd.Timestamp("2024-01-01") + pd.to_timedelta(np.sort(rng.integers(0, 640, n)), "D"),
        "distance_mi": rng.uniform(50, 2800, n).round(0),
        "weight_lb": rng.lognormal(7, 0.6, n).round(0),
        "carrier": rng.choice(list(CARRIER_DELAY), n, p=[0.4, 0.3, 0.2, 0.1]),
        "service": rng.choice(["Standard", "Expedited"], n, p=[0.8, 0.2]),
    })
    month = df.ship_date.dt.month
    # 🐍 PY: np.select picks a label from the first matching condition (vectorised if/elif)
    df["season"] = np.select([month.isin([11, 12]), month.isin([1, 2]), month.isin([3, 4, 5])],
                             ["Peak", "Winter", "Spring"], default="Summer")
    # 🐍 PY: np.where(cond, a, b) is a vectorised if/else over a whole column
    speed = np.where(df.service == "Expedited", 650, 450)
    # Transit moves in whole "driving days" (step-shaped, not a straight line) - k-NN handles steps well
    df["transit_days"] = (1 + np.ceil(df.distance_mi / speed) + 0.00015 * df.weight_lb
                          + df.carrier.map(CARRIER_DELAY) + df.season.map(SEASON_DELAY)
                          + rng.gamma(2, 0.3, n)).clip(1).round(1)
    return df


def build_pipeline():
    # 🧠 ML: one-hot turns carrier/season into 0/1 columns so they can take part in distance calculations
    pre = ColumnTransformer([
        ("num", StandardScaler(), NUM),
        ("cat", OneHotEncoder(handle_unknown="ignore"), CAT),
    ])
    return Pipeline([("pre", pre), ("knn", KNeighborsRegressor())])


df = make_shipments()
# 🧠 ML: time-ordered split - train on the past, test on the most recent 20% (how it is used live)
# 🐍 PY: iloc slicing by position works because rows are already sorted by ship_date
cut = int(len(df) * 0.8)
train, test = df.iloc[:cut], df.iloc[cut:]
print(f"Train {train.ship_date.min().date()} -> {train.ship_date.max().date()} ({len(train)} rows)")
print(f"Test  {test.ship_date.min().date()} -> {test.ship_date.max().date()} ({len(test)} rows)\n")

# Tune k, weighting and distance metric with 5-fold CV
# 🧠 ML: weights='distance' lets close neighbours count more; p=1 Manhattan, p=2 Euclidean
grid = {"knn__n_neighbors": [3, 7, 15, 30], "knn__weights": ["uniform", "distance"], "knn__p": [1, 2]}
search = GridSearchCV(build_pipeline(), grid, cv=KFold(5, shuffle=True, random_state=42),
                      scoring="neg_mean_absolute_error", n_jobs=1)
search.fit(train[NUM + CAT], train.transit_days)
print(f"Best params: {search.best_params_}  CV MAE {-search.best_score_:.3f} days")
cv = pd.DataFrame(search.cv_results_)
# 🐍 PY: pivot_table turns long CV rows into a k x weights table
cv["mae"] = -cv.mean_test_score
print(cv[cv.param_knn__p == search.best_params_["knn__p"]]
      .pivot_table(index="param_knn__n_neighbors", columns="param_knn__weights", values="mae")
      .round(3).to_string(), "\n")

# Compare with baselines on the held-out future period
# 🧠 ML: best_estimator_ is the pipeline refit on ALL training data with the winning settings
best = search.best_estimator_
# 🧠 ML: a linear model is a strong baseline - k-NN must earn its place by beating it
lin = Pipeline([("pre", build_pipeline().named_steps["pre"]), ("lr", LinearRegression())])
models = {"Mean baseline": DummyRegressor(), "Linear regression": lin, "Tuned k-NN": best}
print("Model               | Test MAE (days) | within +/-1 day")
for name, m in models.items():
    m.fit(train[NUM + CAT], train.transit_days)
    p = m.predict(test[NUM + CAT])
    # 🧠 ML: share within +/- 1 day is a business-friendly accuracy metric customers understand
    hit = np.mean(np.abs(p - test.transit_days) <= 1)
    print(f"{name:<19} |     {mean_absolute_error(test.transit_days, p):.3f}       |   {hit:.0%}")

# Global explainability
# 🧠 ML: permutation importance = how much worse MAE gets when one column is shuffled
imp = permutation_importance(best, test[NUM + CAT], test.transit_days,
                             scoring="neg_mean_absolute_error", n_repeats=5, random_state=42)
print("\nPermutation importance (MAE increase):")
for col, v in sorted(zip(NUM + CAT, imp.importances_mean), key=lambda t: -t[1]):
    print(f"  {col:<12} {v:.3f} " + "#" * int(v * 20))


def explain(model, shipment, history, k=5):
    """Show the actual past shipments the model leaned on - k-NN's natural explanation."""
    # 🐍 PY: named_steps is a dict-like view of the pipeline's stages by name
    pre, knn = model.named_steps["pre"], model.named_steps["knn"]
    # 🧠 ML: transform the query with the SAME fitted preprocessor, then ask for its neighbours
    dist, idx = knn.kneighbors(pre.transform(shipment), n_neighbors=k)
    nbrs = history.iloc[idx[0]][NUM + CAT + ["transit_days"]].copy()
    # 🧠 ML: showing neighbour distance tells the user how similar the evidence really is
    nbrs["distance"] = dist[0].round(3)
    return nbrs


new = pd.DataFrame([{"distance_mi": 1800, "weight_lb": 900, "carrier": "CarrierC",
                     "season": "Peak", "service": "Standard"}])
eta = best.predict(new)[0]
print(f"\nNew shipment: 1800 mi, 900 lb, CarrierC, Peak, Standard -> ETA {eta:.1f} days")
print(explain(best, new, train).to_string(index=False))

# Business decision: quote a buffered promise date
# 🧠 ML: use the error distribution on test data to set a buffer that meets a 90% on-time target
resid = test.transit_days - best.predict(test[NUM + CAT])
# 🐍 PY: np.quantile returns the value below which 90% of the residuals fall
buffer = np.quantile(resid, 0.90)
print(f"\nQuote ETA + {buffer:.1f} days to keep ~90% of promises -> promise {np.ceil(eta + buffer):.0f} days")

path = os.path.join(tempfile.mkdtemp(), "transit_knn.joblib")
joblib.dump(best, path)
print(f"Saved pipeline -> {os.path.basename(path)}")
print("Governance: k-NN stores the training data itself - purge per data-retention rules and "
      "re-fit monthly as carrier performance drifts.")

**Try this next:**
- Add 'algorithm': ['ball_tree','brute'] to the grid and time the search.
- Increase CARRIER_DELAY for CarrierB to 2.0 only in the last 300 rows to simulate drift and see test MAE rise.
- Change the buffer quantile to 0.95 and see how the promise date moves.